# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1_sJaD57gHuiz1joEgl4B-u0aDy8jtMDo)



## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


In [108]:
import numpy as np

class WarehouseMDP:
    def __init__(self):
        self.height = 5
        self.width = 6

        # TODO: completa a partir de la imagen
        self.start = (0,0)
        self.walls = {(0,3),(1,1),(2,4),(4,2)}
        self.slippery_states = {(1,2),(2,1),(3,3)}

        self.terminal_states = {
             (0,5): 10.0, (2,2): 2.0, (3,5): -10.0
        }

        self.danger_states = {
          (1,4): -3.0, (4,1): -3.0
        }

        self.living_reward = -1.0
        self.gamma = 0.9


        self.actions = [
            (-1, 0),  # UP
            ( 1, 0),  # DOWN
            ( 0,-1),  # LEFT
            ( 0, 1),  # RIGHT
        ]

    def is_valid_state(self, state):
        row, col = state
        if not (0 <= row < self.height and 0 <= col < self.width):
            return False
        return state not in self.walls

    def states(self):
        return [
            (row, col)
            for row in range(self.height)
            for col in range(self.width)
            if self.is_valid_state((row, col))
        ]

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        if self.is_terminal(state):
            return self.terminal_states[state]
        if state in self.danger_states:
            return self.danger_states[state]
        return self.living_reward

    def get_transition_probs(self, state, action):
        """
        [(next_state, probability), ...] を返す
        """
        if self.is_terminal(state):
            return [(state, 1.0)]

        if state in self.slippery_states:

            perp1 = (action[1], action[0])
            perp2 = (-action[1], -action[0])
            outcomes = [
                (action, 0.8),
                (perp1, 0.1),
                (perp2, 0.1),
            ]
        else:

            outcomes = [(action, 1.0)]

        transitions = []
        for act, prob in outcomes:
            next_state = (
                state[0] + act[0],
                state[1] + act[1],
            )

            if not self.is_valid_state(next_state):
                next_state = state

            transitions.append((next_state, prob))

        return transitions


### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [109]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")


Número de estados: 26
✓ Todas las distribuciones de transición suman 1.



## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [110]:
def expected_next_value(grid, state, action, V):
    return sum(
        prob * V[next_state]
        for next_state, prob in grid.get_transition_probs(state, action)
    )

def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    V = {state: 0.0 for state in grid.states()}
    deltas = []

    for iteration in range(max_iter):
        V_new = V.copy()
        biggest_change = 0.0

        for state in grid.states():
            if grid.is_terminal(state):
                V_new[state] = grid.get_reward(state)
            else:
                best_expected_value = max(
                    expected_next_value(grid, state, action, V)
                    for action in grid.actions
                )
                V_new[state] = (
                    grid.get_reward(state)
                    + grid.gamma * best_expected_value
                )

            biggest_change = max(
                biggest_change,
                abs(V_new[state] - V[state])
            )

        V = V_new
        deltas.append(biggest_change)

        if biggest_change < threshold:
            break

    return V, iteration + 1, deltas

def extract_policy(grid, V):
    policy = {}
    for state in grid.states():
        if grid.is_terminal(state):
            continue
        policy[state] = max(
            grid.actions,
            key=lambda action: expected_next_value(grid, state, action, V)
        )
    return policy

def print_policy(grid, policy):
    for state, action in policy.items():
        print(f"State {state}: Action {action}")

if __name__ == "__main__":

    V_star, iters, deltas = value_iteration(grid)

    policy_star = extract_policy(grid, V_star)
    print("π*(s):")
    print_policy(grid, policy_star)


π*(s):
State (0, 0): Action (0, 1)
State (0, 1): Action (0, 1)
State (0, 2): Action (1, 0)
State (0, 4): Action (0, 1)
State (1, 0): Action (1, 0)
State (1, 2): Action (1, 0)
State (1, 3): Action (0, 1)
State (1, 4): Action (-1, 0)
State (1, 5): Action (-1, 0)
State (2, 0): Action (0, 1)
State (2, 1): Action (0, 1)
State (2, 3): Action (-1, 0)
State (2, 5): Action (-1, 0)
State (3, 0): Action (0, 1)
State (3, 1): Action (0, 1)
State (3, 2): Action (-1, 0)
State (3, 3): Action (-1, 0)
State (3, 4): Action (0, -1)
State (4, 0): Action (-1, 0)
State (4, 1): Action (-1, 0)
State (4, 3): Action (-1, 0)
State (4, 4): Action (-1, 0)
State (4, 5): Action (0, -1)



## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$

In [111]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    V = {state: 0.0 for state in grid.states()}

    for iteration in range(max_iter):
        V_new = V.copy()
        biggest_change = 0.0

        for state in grid.states():
            if grid.is_terminal(state):
                V_new[state] = grid.get_reward(state)
            else:
                action = policy[state]
                V_new[state] = (
                    grid.get_reward(state)
                    + grid.gamma * expected_next_value(grid, state, action, V)
                )

            biggest_change = max(biggest_change, abs(V_new[state] - V[state]))

        V = V_new
        if biggest_change < threshold:
            break

    return V


def policy_improvement(grid, V):
    new_policy = {}

    for state in grid.states():
        if grid.is_terminal(state):
            continue

        new_policy[state] = max(
            grid.actions,
            key=lambda action: expected_next_value(grid, state, action, V)
        )

    return new_policy


def policy_iteration(grid, threshold=1e-4, max_iter=100):
    initial_action = grid.actions[0]
    policy = {
        state: initial_action
        for state in grid.states()
        if not grid.is_terminal(state)
    }
    history = []

    for iteration in range(max_iter):

        V = policy_evaluation(grid, policy, threshold=threshold)

        new_policy = policy_improvement(grid, V)

        changed = sum(
            new_policy[state] != policy[state]
            for state in new_policy
        )

        policy = new_policy
        history.append(changed)

        if changed == 0:
            break

    V = policy_evaluation(grid, policy, threshold=threshold)

    return policy, V, history


## Parte 4 — Visualización y comparación


In [112]:
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [113]:
# VALUE ITERATION
V_vi, n_vi,_ = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n✓ Ambos algoritmos encontraron la misma política óptima.")


=== VALUE ITERATION ===
Iteraciones: 20

Valores:
 -2.240 |  -1.378 |  -0.420 |   WALL   |  +8.000 | +10.000
 -2.295 |   WALL   |  +0.645 |  +2.780 |  +4.200 |  +8.000
 -1.438 |  -0.487 |  +2.000 |  +1.502 |   WALL   |  +6.200
 -1.252 |  -0.280 |  +0.800 |  -0.678 |  -1.610 | -10.000
 -2.127 |  -3.252 |   WALL   |  -1.610 |  -2.449 |  -3.204

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  ↑  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ↑  |  ← 

=== POLICY ITERATION ===
Historia: [13, 7, 3, 1, 0]

Valores:
 -2.240 |  -1.378 |  -0.420 |   WALL   |  +8.000 | +10.000
 -2.295 |   WALL   |  +0.645 |  +2.780 |  +4.200 |  +8.000
 -1.438 |  -0.487 |  +2.000 |  +1.502 |   WALL   |  +6.200
 -1.252 |  -0.280 |  +0.800 |  -0.678 |  -1.610 | -10.000
 -2.127 |  -3.252 |   WALL   |  -1.610 |  -2.449 |  -3.204

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  ↑  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  


## Parte 5 — Interpreta la política

Antes de cambiar parámetros, responde:

1. Desde `START`, ¿el robot busca la **entrega +10** o prefiere la **estación de carga +2**?

A. El robot busca la entrega +10

2. ¿Por qué una recompensa menor podría ser óptima?

A. Porque alcanzar una recompensa mayor (como $+10$) requiere dar más pasos, acumulando más costo por paso ($-1$) y atravesando estados resbaladizos o riesgosos. Si el costo por paso es alto o el camino a $+10$ es muy peligroso, la recompensa cercana de $+2$ puede ofrecer un valor esperado (esperanza) total superior.

3. ¿En qué estados el piso resbaloso cambia la decisión?

A. En los estados resbaladizos como $(1,2)$, $(2,1)$ y $(3,3)$, o en sus estados adyacentes. La probabilidad del $20\%$ de desviarse hace que el robot evite pasar cerca de los peligros ($-3$ y $-10$) para no caer en ellos por accidente, cambiando su dirección hacia rutas más seguras.

4. ¿Qué papel cumple el costo por paso `-1`?

A. Actúa como una penalización por tiempo o retraso (living_reward). Incentiva al robot a llegar a un estado terminal positivo lo antes posible y por el camino más corto, evitando dar vueltas innecesarias en el mapa.

5. ¿Por qué \(T(s,a,s')\) ya no puede implementarse con las mismas probabilidades para todos los estados?

Porque la dinámica del entorno depende del tipo de terreno del estado actual. Los estados de piso normal tienen una probabilidad del $90\%$ (o $100\%$) de éxito en la acción, mientras que los estados de piso resbaloso tienen una menor probabilidad de éxito ($60\%$ u $80\%$) y mayor desvío hacia los lados.
### Experimento A — Menos costo por paso

Cambia:

```python
living_reward = -0.1
```

Predice la política **antes de ejecutar**.

### Experimento B — Piso muy resbaloso

Cambia la probabilidad de movimiento deseado del piso resbaloso:

```python
0.60 → 0.40
```

y reparte el restante entre las dos desviaciones.

### Experimento C — Más paciencia

Cambia:

```python
gamma = 0.99
```

¿La política valora más la recompensa `+10` distante?

### Bonus

Encuentra aproximadamente el valor de `living_reward` a partir del cual la política desde `START` cambia entre:

- ir a carga `+2`;
- intentar llegar a entrega `+10`.


Las respueta de Experimento

ExprimentoA

Predicción:
Creo que esto nos ayudará a asegurar la entrega +10 más que antes.

Resultad:
Al reducir el costo por paso a $-0.1$, el robot no se siente tan penalizado por dar pasos adicionales. Esto refuerza la decisión de ir hacia $+10$ y le permite tomar rutas más largas pero más seguras (alejadas de los peligros).

ExperimentoB

Resultad: Al aumentar el deslizamiento, el riesgo de caer por accidente en estados de peligro ($-3$ o $-10$) se incrementa significativamente. La política cambia para alejarse aún más de las celdas peligrosas y busca la carga +2

ExperimentoC

Resultad: Al aumentar $\gamma$ a $0.99$, el robot le da casi el mismo valor a las recompensas lejanas que a las inmediatas. Esto incentiva fuertemente la búsqueda de la recompensa mayor de $+10$, haciendo que los pasos de distancia importen mucho menos.

Bunos
Resultad: Con un valor de -1.33, se empezó a buscar la carga +2 por primera vez.



.